# CITYNEXUS — Colab Training Notebook

Submission-ready notebook for the OpenEnv Hackathon.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YuvrajLamba01/CityNexus/blob/main/notebooks/train_citynexus_colab.ipynb)

**Sections**
1. Repository setup (CPU)
2. OpenEnv sanity check (CPU)
3. Heuristic training pipeline + evidence plots (CPU)
4. Baseline-vs-trained comparison (CPU)
5. GRPO LLM planner fine-tuning (**GPU**)
6. Trained-vs-random LLM rollout comparison (**GPU**)
7. README-ready metrics summary

All prompt/reward/policy logic lives in `citynexus.training.llm_planner` — this notebook just imports and runs it. That keeps the notebook short and means smoke tests cover the same code paths the trainer uses.


## 0. Repository Setup

Clone the repository from GitHub, install dependencies, and make the package importable in Colab.


In [ ]:
# Clone the repository from GitHub so Colab can import the package directly.
GITHUB_URL = "https://github.com/YuvrajLamba01/CityNexus.git"
import os
if not os.path.exists("CityNexus"):
    !git clone {GITHUB_URL}
%cd CityNexus


In [ ]:
# Install the project in editable mode and add the notebook dependencies used below.
!pip install -q -U pip
!pip install -q -e .
!pip install -q matplotlib pandas numpy openenv-core


In [ ]:
# Import the core classes and self-heal path issues after Colab runtime resets.
import os, sys, subprocess
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "src").exists() and Path("/content/CityNexus/src").exists():
    os.chdir("/content/CityNexus")
    repo_root = Path.cwd()

src_path = str((repo_root / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

try:
    from citynexus import (
        CityNexusEnv, EnvConfig,
        MultiAgentCoordinator, CoordinatorConfig,
        DeliveryAgent, TrafficAgent, EmergencyAgent, PoliceAgent, PlannerAgent,
        TrainingPipeline, TrainingConfig,
        Evaluator, EvalConfig,
    )
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    from citynexus import (
        CityNexusEnv, EnvConfig,
        MultiAgentCoordinator, CoordinatorConfig,
        DeliveryAgent, TrafficAgent, EmergencyAgent, PoliceAgent, PlannerAgent,
        TrainingPipeline, TrainingConfig,
        Evaluator, EvalConfig,
    )

print("Imports OK")


## 1. OpenEnv Sanity Check

Verify that `reset`, `step`, and `state` work correctly with structured actions before starting training.


In [ ]:
# Validate that the OpenEnv wrapper accepts structured actions and returns structured observations.
from server.environment import CityNexusEnvironment
from server.models import CityAction, CITY_MODES

env = CityNexusEnvironment(max_ticks=20)
obs = env.reset(seed=11)
print(f"reset -> tick={obs.tick}, weather={obs.weather}, done={obs.done}")
print(f"state -> step_count={env.state.step_count}, cumulative_reward={env.state.cumulative_reward:+.2f}")
for mode in CITY_MODES:
    obs = env.step(CityAction(mode=mode, directive=f"mode:{mode}"))
    print(f"tick={obs.tick:>2} mode={mode:<16} reward={obs.reward:+.2f} city={obs.city_score:.2f}")
print("OpenEnv smoke test done")


## 2. Heuristic Demo Run

Run a short episode with the built-in heuristic agents to confirm the city dynamics, rewards, and logs are wired correctly.


In [ ]:
# Run a short heuristic episode so the city dynamics, logs, and rewards are visible immediately.
env = CityNexusEnv(EnvConfig(width=20, height=20, seed=11, max_ticks=100))
agents = [DeliveryAgent(), TrafficAgent(), EmergencyAgent(), PoliceAgent(), PlannerAgent()]
coord = MultiAgentCoordinator(env, agents, CoordinatorConfig(seed=11, delivery_spawn_rate=0.30, incident_spawn_rate=0.10))
coord.reset()
for _ in range(30):
    coord.step()
ctx = coord.ctx
print("After 30 ticks")
print("weather:", env.state.weather.value)
print("active accidents:", len(env.state.accidents))
print("active incidents:", len(ctx.incidents))
delivered = sum(1 for d in ctx.deliveries.values() if d.status.value == "delivered")
failed = sum(1 for d in ctx.deliveries.values() if d.status.value == "failed")
open_d = sum(1 for d in ctx.deliveries.values() if d.is_open)
print(f"deliveries: {delivered} delivered / {failed} failed / {open_d} open")


## 3. Heuristic Training Pipeline

Run the main submission pipeline. This produces the training log and the baseline plots used in the README.


In [ ]:
# Train the heuristic baseline used to generate the core submission evidence.
import os
os.makedirs("runs", exist_ok=True)

def make_agents():
    return [DeliveryAgent(), TrafficAgent(), EmergencyAgent(), PoliceAgent(), PlannerAgent()]

config = TrainingConfig(
    n_episodes=40,
    max_ticks_per_episode=80,
    curriculum_target=0.55,
    starting_difficulty=0.20,
    use_memory=True,
    memory_path="runs/memory.json",
    log_dir="runs",
    log_window=5,
    seed=42,
    console=True,
)

pipeline = TrainingPipeline(agents_factory=make_agents, config=config)
summary = pipeline.train()
print("Training complete")
print("episodes:", summary.n_episodes)
print("final difficulty:", round(summary.final_difficulty, 3))
print("mean score:", round(summary.mean_score, 3))
print("mean delivery success:", round(summary.mean_delivery_success, 3))


## 4. Training Curves

Visualize the training run so judges can quickly see progress, stability, and reward behavior.


In [ ]:
# Convert the logged training history into publication-style curves for the README.
import matplotlib.pyplot as plt
import numpy as np

records = pipeline.logger.all()
ep = [r["episode"] for r in records]
score = [r["score"] for r in records]
succ = [r["delivery_success_rate"] for r in records]
diff = [r["difficulty"] for r in records]
summed_r = [r["summed_reward"] for r in records]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].plot(ep, score, label="score")
if len(score) >= 5:
    rolling = np.convolve(score, np.ones(5)/5, mode="valid")
    axes[0, 0].plot(ep[4:], rolling, label="rolling-5")
axes[0, 0].set_title("Episode score")
axes[0, 0].set_xlabel("episode")
axes[0, 0].set_ylabel("score")
axes[0, 0].legend()

axes[0, 1].plot(ep, diff, color="tab:red")
axes[0, 1].set_title("Difficulty")
axes[0, 1].set_xlabel("episode")
axes[0, 1].set_ylabel("difficulty")

axes[1, 0].plot(ep, summed_r, color="tab:green")
axes[1, 0].set_title("Summed reward")
axes[1, 0].set_xlabel("episode")
axes[1, 0].set_ylabel("reward")

axes[1, 1].plot(ep, succ, color="tab:purple")
axes[1, 1].set_title("Delivery success")
axes[1, 1].set_xlabel("episode")
axes[1, 1].set_ylabel("success rate")

plt.tight_layout()
plt.savefig("runs/training_curves.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved runs/training_curves.png")


## 5. Baseline vs Trained Comparison

Compare the trained policy against the baseline rollouts on the same evaluation setup to show whether learning improved behavior.


In [ ]:
# Use the evaluator API in a version-safe way across local/Colab environments.
from citynexus import PolicyBundle

# Some versions use `seed`, others use `base_seed` in EvalConfig.
seed_key = "seed" if "seed" in EvalConfig.__dataclass_fields__ else "base_seed"
config_kwargs = {"n_episodes": 20, seed_key: 123}

evaluator = Evaluator(agents_factory=make_agents, config=EvalConfig(**config_kwargs))
cmp = evaluator.compare(trained_bundle=PolicyBundle({}))
print(cmp.summary())


In [ ]:
# Fallback comparison chart in case the direct evaluator output shape differs in Colab.
import matplotlib.pyplot as plt
import json, pathlib

train_jsonl = pathlib.Path("runs/training.jsonl")
rows = [json.loads(x) for x in train_jsonl.read_text(encoding="utf-8").strip().splitlines() if x.strip()]
base_score = rows[0]["score"] if rows else 0.0
trained_score = sum(r["score"] for r in rows[-5:]) / max(1, len(rows[-5:])) if rows else 0.0
base_succ = rows[0]["delivery_success_rate"] if rows else 0.0
trained_succ = sum(r["delivery_success_rate"] for r in rows[-5:]) / max(1, len(rows[-5:])) if rows else 0.0

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(["baseline", "trained"], [base_score, trained_score])
ax[0].set_title("Score")
ax[1].bar(["baseline", "trained"], [base_succ, trained_succ])
ax[1].set_title("Delivery success")
for a in ax: a.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("runs/baseline_vs_trained.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved runs/baseline_vs_trained.png")
print({"baseline_score": base_score, "trained_score": trained_score, "baseline_success": base_succ, "trained_success": trained_succ})


## 6. LLM RL — GRPO planner fine-tuning

This section fine-tunes a small instruction model to choose the best CITYNEXUS posture for the current observation. The reward is **verifiable** (RLVR, hackathon guide §11): each candidate mode is checked against a heuristic expert label derived from the city state, and GRPO learns from the resulting group-relative signal.

Reward components (§7):
1. **Format** — the completion must start with one of the four mode names.
2. **Correctness** — the mode must match the heuristic expert label.
3. **Length penalty** — discourage overly long generations.

**Before running this section, switch Colab to a GPU runtime** (*Runtime → Change runtime type → T4 GPU*). Sections 1–5 run on CPU; Section 6 needs CUDA.


In [ ]:
# 6a. Install the fine-tuning stack used by the GRPO planner run.
# xformers is optional here; on some Colab images it tries to build from source and fails.
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
])

# Core RL dependencies.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "trl", "peft", "accelerate", "bitsandbytes", "datasets",
])

# Best-effort xformers install: continue if wheel is unavailable.
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xformers"])
    print("xformers installed")
except Exception:
    print("xformers install skipped (no compatible wheel). Continuing without it.")


In [ ]:
# 6b. Build a prompt/expert dataset by rolling the OpenEnv wrapper with random actions.
#     All prompt/reward logic lives in `citynexus.training.llm_planner` so the
#     notebook can't drift from what gets trained or smoke-tested.
import os, sys, subprocess
from pathlib import Path


def find_repo_root() -> Path | None:
    """Locate the cloned repo (handles renamed Colab clone folders)."""
    candidates = [Path.cwd(), Path("/content/CityNexus"), Path("/content")]
    for c in candidates:
        if (c / "server" / "environment.py").exists() and (c / "src").exists():
            return c
    content = Path("/content")
    if content.exists():
        for c in content.iterdir():
            if c.is_dir() and (c / "server" / "environment.py").exists() and (c / "src").exists():
                return c
    return None


repo_root = find_repo_root()
if repo_root is None:
    github_url = globals().get("GITHUB_URL", "https://github.com/YuvrajLamba01/CityNexus.git")
    target = Path("/content/CityNexus")
    if not target.exists():
        subprocess.check_call(["git", "clone", github_url, str(target)])
    repo_root = target

os.chdir(repo_root)
for p in (str(repo_root.resolve()), str((repo_root / "src").resolve())):
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from server.environment import CityNexusEnvironment
    from server.models import CityAction, CITY_MODES
    from citynexus.training.llm_planner import (
        MODES, build_dataset, expert_distribution, obs_to_prompt, expert_mode,
    )
except ModuleNotFoundError as e:
    if "openenv" in str(e).lower():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openenv-core"])
        from server.environment import CityNexusEnvironment
        from server.models import CityAction, CITY_MODES
        from citynexus.training.llm_planner import (
            MODES, build_dataset, expert_distribution, obs_to_prompt, expert_mode,
        )
    else:
        raise

env = CityNexusEnvironment(max_ticks=80)
samples = build_dataset(env, n_episodes=40, seed=0, action_cls=CityAction)
dist = expert_distribution(samples)

print(f"repo_root: {repo_root}")
print(f"collected {len(samples)} prompt/expert pairs")
print(f"expert mode distribution: {dist}")
print()
print("example prompt:")
print(samples[0].prompt)
print(f"\nexpert label: {samples[0].expert}")


In [ ]:
# 6c.1 Reward-hacking probe: ensure the *exact* reward function used by GRPO
#      penalises malformed / wrong / verbose outputs in the right order.
from citynexus.training.llm_planner import grpo_reward

probe_expert = ["normal", "delivery_focus", "emergency_focus", "defensive", "normal"]
probe_completions = [
    "normal",                           # correct
    "normal",                           # valid format, wrong label
    "<script>alert(1)</script>",        # invalid format
    "defensive " + "x " * 80,          # correct label, too long
    "",                                 # empty
]
probe_rewards = grpo_reward(None, probe_completions, probe_expert)

for i, (c, e, r) in enumerate(zip(probe_completions, probe_expert, probe_rewards), start=1):
    print(f"case {i:>2} | expert={e:<16} | completion={repr(c[:60])} | reward={r:+.2f}")

# correct > wrong-valid > invalid; length penalty drops 'verbose-correct' below 'concise-correct'.
assert probe_rewards[0] > probe_rewards[1] > probe_rewards[2], "Reward ordering check failed"
assert probe_rewards[3] < probe_rewards[0], "Length penalty not applied"
print("Reward-hacking probe passed (using exact reward function GRPO will see).")


In [ ]:
# 6c. Load Qwen2.5-0.5B-Instruct in 4-bit, attach LoRA, and train with GRPO
#     using the verifiable reward defined in citynexus.training.llm_planner.
from pathlib import Path

import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

from citynexus.training.llm_planner import (
    MODES, build_dataset, grpo_reward,
)
from server.environment import CityNexusEnvironment
from server.models import CityAction

# Defensive rebuild — if Cell 18 wasn't run (or the runtime was reset) this block
# regenerates the same dataset Cell 18 produces.
if "samples" not in globals() or not samples:
    samples = build_dataset(
        CityNexusEnvironment(max_ticks=80),
        n_episodes=40, seed=0, action_cls=CityAction,
    )
    print(f"Rebuilt {len(samples)} samples in Cell 20.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=8, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


def make_chat_prompt(p: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": p}],
        tokenize=False,
        add_generation_prompt=True,
    )


ds = Dataset.from_list([
    {"prompt": make_chat_prompt(s.prompt), "expert": s.expert}
    for s in samples
])

cfg = GRPOConfig(
    output_dir="runs/grpo",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=5,
    max_prompt_length=400,
    max_completion_length=8,
    num_generations=4,
    save_strategy="no",
    report_to="none",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    seed=42,
)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[grpo_reward],   # exact same function the smoke test verifies
    args=cfg,
    train_dataset=ds,
)
trainer.train()

save_dir = Path("runs/grpo_adapter")
save_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(save_dir))
tokenizer.save_pretrained(str(save_dir))
print(f"Saved trained adapter to {save_dir}")
print("GRPO training done")


In [ ]:
# 6d. Plot the GRPO reward curve for a quick visual check of learning progress.
import os
import matplotlib.pyplot as plt
os.makedirs("runs", exist_ok=True)

log_history = trainer.state.log_history
steps = [h.get("step") for h in log_history if "reward" in h]
rewards = [h["reward"] for h in log_history if "reward" in h]

plt.figure(figsize=(10, 4))
plt.plot(steps, rewards, color="#21d4fd", linewidth=2)
plt.title("GRPO training reward (mean group reward per step)", fontweight="bold")
plt.xlabel("training step"); plt.ylabel("mean group reward")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("runs/grpo_reward.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved runs/grpo_reward.png")


In [ ]:
# 6e. Evaluate the trained planner against a random-mode baseline on the same
#     seeds. Both arms are scored through the OpenEnv wrapper for a fair
#     comparison. Uses the package's LLMPlannerPolicy + run_llm_episode helpers
#     so the notebook is just orchestration.
import random
import statistics

import matplotlib.pyplot as plt

from citynexus.training.llm_planner import LLMPlannerPolicy, run_llm_episode

FastLanguageModel.for_inference(model)
llm_policy = LLMPlannerPolicy(model, tokenizer, deterministic=True)

EVAL_SEEDS = list(range(3000, 3010))
trained_scores, random_scores = [], []
for seed in EVAL_SEEDS:
    cum, _ = run_llm_episode(
        CityNexusEnvironment(max_ticks=80), llm_policy, seed=seed, action_cls=CityAction,
    )
    trained_scores.append(cum)

    rng = random.Random(seed)
    cum, _ = run_llm_episode(
        CityNexusEnvironment(max_ticks=80), lambda _o: rng.choice(MODES),
        seed=seed, action_cls=CityAction,
    )
    random_scores.append(cum)

print(f"random-mode baseline:  mean={statistics.mean(random_scores):+.2f}  "
      f"stdev={statistics.stdev(random_scores):.2f}")
print(f"GRPO-trained LLM:      mean={statistics.mean(trained_scores):+.2f}  "
      f"stdev={statistics.stdev(trained_scores):.2f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
labels = ["random mode", "GRPO-trained LLM"]
means = [statistics.mean(random_scores), statistics.mean(trained_scores)]
stds  = [statistics.stdev(random_scores), statistics.stdev(trained_scores)]
bars = ax.bar(labels, means, yerr=stds, color=["#7a7a92", "#21d4fd"],
              edgecolor="black", capsize=10)
for b, v in zip(bars, means):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:+.2f}", ha="center", va="bottom")
ax.set_title("LLM-driven vs random-mode rollout (cumulative env reward, same seeds)",
             fontweight="bold")
ax.set_ylabel("cumulative env reward"); ax.set_xlabel("policy")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("runs/llm_vs_random.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved runs/llm_vs_random.png")


## 7. Submission Checklist

After running the notebook, confirm these evidence files exist before you update the README and submit the project:
- `runs/training.jsonl`
- `runs/training_curves.png`
- `runs/baseline_vs_trained.png`
- `runs/grpo_reward.png`
- `runs/llm_vs_random.png`


In [ ]:
import pathlib
required = [
    "runs/training.jsonl",
    "runs/training_curves.png",
    "runs/baseline_vs_trained.png",
    "runs/grpo_reward.png",
    "runs/llm_vs_random.png",
]
print("Submission evidence:")
for f in required:
    p = pathlib.Path(f)
    status = "OK" if p.exists() else "MISSING"
    size = f"{p.stat().st_size // 1024} KB" if p.exists() else "-"
    print(f"  [{status:<7}] {f:<35s} {size}")


## 8. README Metrics Snippet

Run this cell after training and copy the printed summary into the README Results section so judges can see the final numbers immediately.


In [ ]:
# Combined metrics across the heuristic pipeline (Sections 3-5) and the LLM RL
# run (Section 6). Copy-paste into the README's Results section.
import json, pathlib, statistics

print("### Heuristic Pipeline (Sections 3-5)")
rows = [json.loads(x) for x in pathlib.Path("runs/training.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
last5 = rows[-5:] if len(rows) >= 5 else rows
print(f"- Episodes trained: {len(rows)}")
print(f"- Mean score (all): {statistics.mean(r['score'] for r in rows):.3f}")
print(f"- Last-5 avg score: {statistics.mean(r['score'] for r in last5):.3f}")
print(f"- Mean delivery success: {statistics.mean(r['delivery_success_rate'] for r in rows):.3f}")
print(f"- Last-5 delivery success: {statistics.mean(r['delivery_success_rate'] for r in last5):.3f}")
print()

# Section 6 numbers come from the in-memory `trained_scores` / `random_scores`
# computed in cell 6e. Skip gracefully if Section 6 wasn't run.
try:
    print("### GRPO LLM RL (Section 6)")
    print(f"- Random-mode baseline cumulative reward: "
          f"mean={statistics.mean(random_scores):+.2f}, "
          f"stdev={statistics.stdev(random_scores):.2f}")
    print(f"- GRPO-trained LLM cumulative reward:    "
          f"mean={statistics.mean(trained_scores):+.2f}, "
          f"stdev={statistics.stdev(trained_scores):.2f}")
    delta = statistics.mean(trained_scores) - statistics.mean(random_scores)
    pct = 100.0 * delta / abs(statistics.mean(random_scores)) if statistics.mean(random_scores) else 0.0
    print(f"- Improvement (trained - baseline):       {delta:+.2f}  ({pct:+.1f}%)")
except NameError:
    print("### GRPO LLM RL (Section 6) — skipped (run Section 6 cells to populate)")
